In [ ]:
import numpy as np
import pandas as pd
from patsy import dmatrices, build_design_matrices
import statsmodels.api as sm

# ---- Load data ----
csv_path = "09_expected-points.csv"  # update if saved elsewhere
df = pd.read_csv(csv_path)

# ---- Model spec ----
formula = (
    "pts_next_score ~ "
    "bs(yardline_100, df=5, degree=3) + "
    "C(down) + "
    "np.log(ydstogo) + "
    "bs(half_seconds_remaining, df=5, degree=3)"
)

y, X = dmatrices(formula, data=df, return_type="dataframe")

# MNLogit needs the response as integer category codes (0..K-1)
outcome_levels = sorted(df["pts_next_score"].unique())
level_to_code = {lvl: i for i, lvl in enumerate(outcome_levels)}
y_codes = df["pts_next_score"].map(level_to_code)

print(f"Fitting multinomial logit on {len(df):,} plays, "
      f"{len(outcome_levels)} outcome categories: {outcome_levels}")

model = sm.MNLogit(y_codes, X)
result = model.fit(method="newton", maxiter=100, disp=True)


# ---- Inspect fit ----
result.summary()

In [ ]:
# ---- Target estimand: 1st-and-10 at opp 35, ydstogo=10, half_seconds_remaining=1800 ----
new_data = pd.DataFrame({
    "yardline_100": [35],
    "down": [1],
    "ydstogo": [10],
    "half_seconds_remaining": [1800],
})

X_new = build_design_matrices([X.design_info], new_data, return_type="dataframe")[0]

# MNLogit predict returns probabilities ordered by integer codes (0..K-1),
# which matches the order of outcome_levels
probs = np.asarray(result.predict(X_new)).flatten()

expected_points = float(np.dot(probs, outcome_levels))

print("Predicted outcome probabilities at the target estimand")
print("(1st-and-10, yardline_100=35, ydstogo=10, half_seconds_remaining=1800):")
for lvl, p in zip(outcome_levels, probs):
    print(f"  pts_next_score = {lvl:>3}: {p:.4f}")

print(f"\nExpected points (target estimand) = {expected_points:.4f}")

In [ ]:
#Bootstrap Choice: I want to use a cluster bootstrap. Observational bootstraps don't work because each observation is not iid.
# Parametric bootstraps don't work because I don't have faith that my multinomial probability model is unbiased.
# Residual bootstraps don't work because I'm not using a linear model to estimate probabilities

import pandas as pd
import numpy as np

# ---- Sort to ensure plays are in chronological order within each game ----
df = df.sort_values(['game_id', 'play_id']).reset_index(drop=True)

# Sign of posteam_spread for each play
sign = np.sign(df['posteam_spread'])

# A new drive starts whenever the sign of posteam_spread changes
# within the same game_id, or whenever the game_id itself changes
new_drive = (sign != sign.groupby(df['game_id']).shift()) | (df['game_id'] != df['game_id'].shift())

# Cumulative sum gives a global, monotonically increasing drive number
df['drive_no'] = new_drive.cumsum()


print(df)

In [ ]:
import numpy as np
import pandas as pd
from patsy import dmatrices, build_design_matrices
import statsmodels.api as sm

# ---- Setup (assumes df already has drive_no from the previous step) ----
n_drives = df['drive_no'].nunique()
drive_ids = df['drive_no'].unique()
drive_groups = {d: g for d, g in df.groupby('drive_no')}

formula = (
    "pts_next_score ~ "
    "bs(yardline_100, df=5, degree=3) + "
    "C(down) + "
    "np.log(ydstogo) + "
    "bs(half_seconds_remaining, df=5, degree=3)"
)

outcome_levels = sorted(df["pts_next_score"].unique())
level_to_code = {lvl: i for i, lvl in enumerate(outcome_levels)}

# Target estimand: 1st-and-10, opp 35, ydstogo=10, half_seconds_remaining=1800
new_data = pd.DataFrame({
    "yardline_100": [35],
    "down": [1],
    "ydstogo": [10],
    "half_seconds_remaining": [1800],
})

n_boot = 200
ep_estimates = []

np.random.seed(42)  # for reproducibility; remove/change as desired

for b in range(n_boot):
    # (1)-(3) sample 15,899 drives (with replacement -> standard block bootstrap)
    sampled_drives = np.random.choice(drive_ids, size=n_drives, replace=True)
    boot_df = pd.concat([drive_groups[d] for d in sampled_drives], ignore_index=True)

    # (4) fit the same multinomial model
    y, X = dmatrices(formula, data=boot_df, return_type="dataframe")
    y_codes = boot_df["pts_next_score"].map(level_to_code)

    model = sm.MNLogit(y_codes, X)
    result = model.fit(method="newton", maxiter=100, disp=False)

    # (5) compute EP for the target estimand
    X_new = build_design_matrices([X.design_info], new_data, return_type="dataframe")[0]
    probs = np.asarray(result.predict(X_new)).flatten()
    ep = float(np.dot(probs, outcome_levels))
    ep_estimates.append(ep)


ep_estimates = np.array(ep_estimates)
print(f"\nMean EP across {n_boot} bootstraps: {ep_estimates.mean():.4f}")
print(f"Std dev: {ep_estimates.std():.4f}")
print(f"95% CI: [{np.percentile(ep_estimates, 2.5):.4f}, {np.percentile(ep_estimates, 97.5):.4f}]")

In [ ]:
#Free Throws

import pandas as pd

# Load the dataset (semicolon-delimited, latin1 encoding)
df = pd.read_csv("08_nba-free-throws.csv", sep=";", encoding="latin1")

# Keep only relevant columns
df = df[["Player", "Tm", "FT", "G", "GS", "FTA"]]

# Drop "TOT" rows for players who have multiple entries (avoid double-counting)
multi_team = df["Player"].duplicated(keep=False)
df = df[~((df["Tm"] == "TOT") & multi_team)]

# Compress multiple rows per player by summing stats
df = df.groupby("Player", as_index=False)[["FT", "G", "GS", "FTA"]].sum()

# Filter for players with (FTA * G) > 25
df = df[(df["FTA"] * df["G"]) > 25].reset_index(drop=True)

# Compute FT_total, FTA_total, FT%
df["FT_total"] = df["FT"] * df["G"]
df["FTA_total"] = df["FTA"] * df["G"]
df["FT%"] = df["FT_total"] / df["FTA_total"]


import numpy as np
import matplotlib.pyplot as plt

z = 1.96  # 95% confidence

# Compute Wald CI
p_hat = df["FT%"]
n = df["FTA_total"]

wald_se = np.sqrt(p_hat * (1 - p_hat) / n)
df["Wald_low"] = p_hat - z * wald_se
df["Wald_high"] = p_hat + z * wald_se

# Compute Agresti-Coull CI
n_tilde = n + z**2
p_tilde = (df["FT_total"] + z**2 / 2) / n_tilde
ac_se = np.sqrt(p_tilde * (1 - p_tilde) / n_tilde)
df["AC_low"] = p_tilde - z * ac_se
df["AC_high"] = p_tilde + z * ac_se

import numpy as np
import matplotlib.pyplot as plt



z = 1.96
B = 20  # number of bootstrap datasets

# Randomly sample 100 players
sample_df = df.sample(n=100, random_state=42).reset_index(drop=True)

boot_low = np.zeros(len(sample_df))
boot_high = np.zeros(len(sample_df))
boot_var = np.zeros(len(sample_df))

for i, row in sample_df.iterrows():
    n_attempts = int(row["FTA_total"])
    n_made = int(row["FT_total"])

    # Original sequence of free throws: 1 = made, 0 = missed
    sequence = np.array([1] * n_made + [0] * (n_attempts - n_made))

    # Generate B bootstrap resamples (with replacement) and compute FT% for each
    boot_phats = np.array([
        np.mean(np.random.choice(sequence, size=n_attempts, replace=True))
        for _ in range(B)
    ])

    boot_var[i] = np.var(boot_phats)
    boot_low[i] = np.percentile(boot_phats, 2.5)
    boot_high[i] = np.percentile(boot_phats, 97.5)

sample_df["Boot_low"] = boot_low
sample_df["Boot_high"] = boot_high
sample_df["Boot_var"] = boot_var

# Wald CI
p_hat = sample_df["FT%"]
n = sample_df["FTA_total"]
wald_se = np.sqrt(p_hat * (1 - p_hat) / n)
sample_df["Wald_low"] = p_hat - z * wald_se
sample_df["Wald_high"] = p_hat + z * wald_se

# Agresti-Coull CI
n_tilde = n + z**2
p_tilde = (sample_df["FT_total"] + z**2 / 2) / n_tilde
ac_se = np.sqrt(p_tilde * (1 - p_tilde) / n_tilde)
sample_df["AC_low"] = p_tilde - z * ac_se
sample_df["AC_high"] = p_tilde + z * ac_se

# Plot all three CIs overlaid
fig, ax = plt.subplots(figsize=(20, 10))
x = np.arange(len(sample_df))
offset = 0.15

ax.errorbar(
    x - offset, sample_df["FT%"],
    yerr=[sample_df["FT%"] - sample_df["Wald_low"], sample_df["Wald_high"] - sample_df["FT%"]],
    fmt="o", color="blue", capsize=3, label="Wald CI"
)

ax.errorbar(
    x, sample_df["FT%"],
    yerr=[sample_df["FT%"] - sample_df["AC_low"], sample_df["AC_high"] - sample_df["FT%"]],
    fmt="o", color="red", capsize=3, label="Agresti-Coull CI"
)

ax.errorbar(
    x + offset, sample_df["FT%"],
    yerr=[sample_df["FT%"] - sample_df["Boot_low"], sample_df["Boot_high"] - sample_df["FT%"]],
    fmt="o", color="green", capsize=3, label="Bootstrap CI"
)

ax.set_xticks(x)
ax.set_xticklabels(sample_df["Player"], rotation=90, fontsize=7)
ax.set_xlabel("Player")
ax.set_ylabel("FT% with 95% CI")
ax.set_title("95% Wald, Agresti-Coull, and Bootstrap CIs for Free Throw Percentage")
ax.legend()
plt.tight_layout()
plt.show()

print(sample_df[["Player", "FT%", "Boot_var", "Wald_low", "Wald_high", "AC_low", "AC_high", "Boot_low", "Boot_high"]])

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

z = 1.96
B = 20  # number of bootstrapped datasets
n_values = [10, 50, 100, 250, 500, 1000]
p_grid = np.linspace(0.01, 0.99, 1000)  # avoid exact 0/1 for SD computation

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for idx, n in enumerate(n_values):
    wald_low = np.zeros(len(p_grid))
    wald_high = np.zeros(len(p_grid))
    ac_low = np.zeros(len(p_grid))
    ac_high = np.zeros(len(p_grid))
    boot_low = np.zeros(len(p_grid))
    boot_high = np.zeros(len(p_grid))

    for j, p in enumerate(p_grid):
        # Generate one sample of n observations ~ Bernoulli(p)
        x = np.random.binomial(n, p)
        p_hat = x / n

        # Wald CI
        wald_se = np.sqrt(p_hat * (1 - p_hat) / n)
        wald_low[j] = p_hat - z * wald_se
        wald_high[j] = p_hat + z * wald_se

        # Agresti-Coull CI
        n_tilde = n + z**2
        p_tilde = (x + z**2 / 2) / n_tilde
        ac_se = np.sqrt(p_tilde * (1 - p_tilde) / n_tilde)
        ac_low[j] = p_tilde - z * ac_se
        ac_high[j] = p_tilde + z * ac_se

        # Bootstrap: resample from the original n observations B times
        boot_phats = np.random.binomial(n, p_hat, size=B) / n
        boot_sd = np.std(boot_phats)
        boot_low[j] = p_hat - z * boot_sd
        boot_high[j] = p_hat + z * boot_sd

    ax = axes[idx]
    ax.plot(p_grid, wald_low, color="blue", linestyle="--", label="Wald CI")
    ax.plot(p_grid, wald_high, color="blue", linestyle="--")
    ax.plot(p_grid, ac_low, color="red", linestyle="--", label="Agresti-Coull CI")
    ax.plot(p_grid, ac_high, color="red", linestyle="--")
    ax.plot(p_grid, boot_low, color="green", linestyle="--", label="Bootstrap CI")
    ax.plot(p_grid, boot_high, color="green", linestyle="--")
    ax.plot(p_grid, p_grid, color="black", linewidth=0.8, label="p (true)")

    ax.set_title(f"n = {n}")
    ax.set_xlabel("p")
    ax.set_ylabel("CI bounds")
    ax.set_ylim(-0.05, 1.05)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

z = 1.96
M = 50   # number of samples per (n, p)
B = 20   # number of bootstrapped datasets per sample
n_values = [10, 50, 100, 250, 500, 1000]
p_grid = np.linspace(0.01, 0.99, 1000)

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for idx, n in enumerate(n_values):
    wald_coverage = np.zeros(len(p_grid))
    ac_coverage = np.zeros(len(p_grid))
    boot_coverage = np.zeros(len(p_grid))

    for j, p in enumerate(p_grid):
        # (1) Simulate M samples of size n
        x = np.random.binomial(n, p, size=M)
        p_hat = x / n

        # (2) Wald CI
        wald_se = np.sqrt(p_hat * (1 - p_hat) / n)
        wald_low = p_hat - z * wald_se
        wald_high = p_hat + z * wald_se
        wald_coverage[j] = np.mean((wald_low <= p) & (p <= wald_high))

        # Agresti-Coull CI
        n_tilde = n + z**2
        p_tilde = (x + z**2 / 2) / n_tilde
        ac_se = np.sqrt(p_tilde * (1 - p_tilde) / n_tilde)
        ac_low = p_tilde - z * ac_se
        ac_high = p_tilde + z * ac_se
        ac_coverage[j] = np.mean((ac_low <= p) & (p <= ac_high))

        # (3) For each sample, generate B bootstrapped datasets
        boot_x = np.random.binomial(n, p_hat[:, None], size=(M, B))
        boot_phat = boot_x / n
        boot_sd = np.std(boot_phat, axis=1)

        boot_low = p_hat - z * boot_sd
        boot_high = p_hat + z * boot_sd
        boot_coverage[j] = np.mean((boot_low <= p) & (p <= boot_high))

    ax = axes[idx]
    ax.plot(p_grid, wald_coverage, label="Wald", color="blue")
    ax.plot(p_grid, ac_coverage, label="Agresti-Coull", color="red")
    ax.plot(p_grid, boot_coverage, label="Bootstrap SD", color="green")
    ax.axhline(0.95, color="black", linestyle="--", linewidth=1)
    ax.set_title(f"n = {n}")
    ax.set_xlabel("p")
    ax.set_ylabel("Coverage Probability")
    ax.set_ylim(0, 1.05)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()